In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')




Mounted at /content/drive


In [ ]:
# Define paths
DIR = "/content/drive/MyDrive/NLP"
MODEL_DIR = f"{DIR}/ngram_model"
OUTPUT_DIR = f"{DIR}/Assignment6"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load necessary libraries
import pickle
import random
import math
from collections import Counter, defaultdict

In [ ]:

def load_model(filename):
    with open(filename, "rb") as f:
        model = pickle.load(f)
    return model

In [ ]:
def load_sentences(filename):
    sentences = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                if line.startswith("[") and line.endswith("]"):
                    sent = line[1:-1].split(", ")
                    sent = [w.strip("'\"") for w in sent]
                    sentences.append(sent)
                else:
                    sentences.append(line.split())
    return sentences



In [ ]:

bi_model=load_model(f'{MODEL_DIR}/final_2gram_counts.pkl')
uni_model=load_model(f'{MODEL_DIR}//final_1gram_counts.pkl')
tri_model=load_model(f'{MODEL_DIR}/final_3gram_counts.pkl')
quad_model=load_model(f'{MODEL_DIR}/final_4gram_counts.pkl')


In [ ]:
test_sentences = load_sentences(f"{MODEL_DIR}/test_sentences.csv")
print(f"Loaded {len(test_sentences)} test sentences.")
validation_sentences=load_sentences(f"{MODEL_DIR}/val_sentences.csv")
print(f"Loaded {len(validation_sentences)} val sentences.")

Loaded 1001 test sentences.
Loaded 1001 val sentences.


In [ ]:
%whos

Variable               Type        Data/Info
--------------------------------------------
Counter                type        <class 'collections.Counter'>
DIR                    str         /content/drive/MyDrive/NLP
MODEL_DIR              str         /content/drive/MyDrive/NLP/ngram_model
OUTPUT_DIR             str         /content/drive/MyDrive/NLP/Assignment6
bi_model               dict        n=5143767
defaultdict            type        <class 'collections.defaultdict'>
drive                  module      <module 'google.colab.dri<...>s/google/colab/drive.py'>
load_model             function    <function load_model at 0x7e289119eb60>
load_sentences         function    <function load_sentences at 0x7e289119eac0>
math                   module      <module 'math' (built-in)>
os                     module      <module 'os' (frozen)>
pickle                 module      <module 'pickle' from '/u<...>ib/python3.12/pickle.py'>
quad_model             dict        n=12034694
random             

# katz_backoff_prob

In [ ]:
def katz_backoff_prob(w1, w2, w3, w4, uni_model, bi_model, tri_model, quad_model, d=0.5):
    # Quadrigram
    quad_count = quad_model.get((w1,w2,w3,w4), 0)
    tri_count  = tri_model.get((w1,w2,w3), 0)

    if quad_count > 0:
        return max(quad_count - d, 0) / tri_count
    else:
        # Backoff weight (α) can be computed but for simplicity, we normalize to trigram prob
        tri_prob = (tri_model.get((w2,w3,w4),0) / max(bi_model.get((w2,w3),1),1))
        return d * tri_prob


In [ ]:
katz_probs = []


for sent in test_sentences:
    sent_probs_katz = []
    sent_probs_kn = []

    # Pad the sentence with start tokens
    padded = ["<s>","<s>","<s>"] + sent + ["</s>"]

    for i in range(3, len(padded)):
        w1, w2, w3, w4 = padded[i-3:i+1]
        sent_probs_katz.append(katz_backoff_prob(w1,w2,w3,w4, uni_model, bi_model, tri_model, quad_model))

    katz_probs.append(sent_probs_katz)

In [ ]:
with open(os.path.join(OUTPUT_DIR,"katz_probs.json"), "w", encoding="utf-8") as f:
    json.dump(katz_probs, f, ensure_ascii=False, indent=4)

print(" Katz probabilities saved")


NameError: name 'json' is not defined